# 02 — Detector YOLOv8n (una clase: "mosquito") — correr en Google Colab

1. `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)`.
2. Subir `data/processed/yolo.zip` (generado por `src/prepare_data.py`) a la raíz de **Mi unidad** en Google Drive.
3. `Entorno de ejecución > Ejecutar todo`. Tarda ~35–45 min.
4. Descargar `yolo_resultados.zip` (queda en Drive) y descomprimirlo en `models/detector/` del repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/yolo.zip -d /content/yolo
# Ultralytics resuelve rutas relativas contra su carpeta de datasets: usar ruta absoluta
open('/content/yolo/data.yaml', 'w').write(
    "path: /content/yolo
train: images/train
val: images/val
test: images/test
names:
  0: mosquito
")
!ls /content/yolo/images/train | wc -l

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # preentrenado en COCO
model.train(data='/content/yolo/data.yaml', imgsz=640, epochs=30, batch=32, patience=8,
            seed=42, project='/content/runs', name='det', exist_ok=True)

In [ ]:
# Métricas estándar en test y criterio del reto: IoU >= 0.75 con la caja de mayor confianza
best = YOLO('/content/runs/det/weights/best.pt')
metrics = best.val(data='/content/yolo/data.yaml', split='test', imgsz=640)
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

import glob, os, csv

def iou(a, b):  # cajas (cx, cy, w, h) normalizadas
    ax1, ay1, ax2, ay2 = a[0]-a[2]/2, a[1]-a[3]/2, a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    return inter / (a[2]*a[3] + b[2]*b[3] - inter + 1e-9)

rows = []
for path in sorted(glob.glob('/content/yolo/images/test/*.jpg')):
    stem = os.path.splitext(os.path.basename(path))[0]
    gt = [float(v) for v in open(f'/content/yolo/labels/test/{stem}.txt').read().split()[1:]]
    r = best.predict(path, imgsz=640, conf=0.001, verbose=False)[0]
    if len(r.boxes):
        k = int(r.boxes.conf.argmax())
        pred = r.boxes.xywhn[k].tolist()
        rows.append([stem, float(r.boxes.conf[k]), *pred, iou(pred, gt)])
    else:
        rows.append([stem, 0.0, 0, 0, 0, 0, 0.0])

with open('/content/runs/det/det_test_boxes.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['img', 'conf', 'cx', 'cy', 'w', 'h', 'iou']); w.writerows(rows)
ious = [r[-1] for r in rows]
print(f'Imágenes de test: {len(ious)} | IoU medio: {sum(ious)/len(ious):.3f} | '
      f'IoU >= 0.75: {sum(i >= 0.75 for i in ious)/len(ious):.1%}')

In [ ]:
import shutil
shutil.make_archive('/content/drive/MyDrive/yolo_resultados', 'zip', '/content/runs/det')
print('Listo: descargar yolo_resultados.zip desde Mi unidad')